In [1]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np
import time

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os  

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

from requests.exceptions import ConnectTimeout, ReadTimeout, RequestException

In [2]:
date = "2025-12"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

In [3]:
end_use = "naics?get=CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url) 
    
print(r)
    
df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

df.columns = r.json()[0]

df["total_imports"] = df["CON_VAL_MO"].astype(float)

df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])

top_products = grp.agg({"total_imports":"sum","CTY_CODE":"first"})

country_list = list(top_products.sort_values(by = "total_imports", ascending = False).CTY_CODE)[0:31]


['TOTAL FOR ALL COUNTRIES','NAFTA','EUROPEAN UNION']

<Response [200]>


['TOTAL FOR ALL COUNTRIES', 'NAFTA', 'EUROPEAN UNION']

In [4]:
df.tail()

,CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL,time,total_imports
39302,7713921,7940,ZAMBIA,DET,2025-12,7713921.0
39303,731477,7950,ESWATINI,DET,2025-12,731477.0
39304,1664689,7960,ZIMBABWE,DET,2025-12,1664689.0
39305,5007915,7970,MALAWI,DET,2025-12,5007915.0
39306,7276008,7990,LESOTHO,DET,2025-12,7276008.0


In [5]:
country_list[0] = ""

In [6]:
country_list.extend(["0003", "0020"])

In [7]:
len(country_list)

33

In [8]:
end_use = "hs?get=CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

surl  = surl + my_key + "&COMM_LVL=HS10" 

for xxx in country_list:
    
    out_file = ".\\data"+ "\\imports-hs10\\" + xxx + "data-" + date + ".parquet"
    
    if xxx == "":
        out_file = ".\\data"+ "\\imports-hs10\\" + "TOTAL" + "data-" + date + ".parquet"
    
    
    if os.path.exists(out_file):
        
        print("Already have downloaded file")
        
        continue
    
    print(f"Downloading {xxx} for {date}")
    
    url = surl + "&time=" + date
    
    if xxx != "":
        url = url + "&CTY_CODE=" + xxx
    
    max_retries = 5
    retry_count = 0
    r = None
    
    while retry_count < max_retries:
        try:
            # Added timeout (30 seconds) and read timeout
            r = requests.get(url, timeout=30)
            
            if r.status_code == 200:
                break
            else:
                print(f"Request failed with status {r.status_code}, waiting 30 seconds...")
                time.sleep(30)
                retry_count += 1
                
        except (ConnectTimeout, ReadTimeout) as e:
            retry_count += 1
            print(f"Connection timeout on attempt {retry_count}/{max_retries}: {type(e).__name__}")
            if retry_count < max_retries:
                wait_time = 30 * (2 ** (retry_count - 1))  # Exponential backoff
                print(f"Waiting {wait_time} seconds before retry...")
                time.sleep(wait_time)
            else:
                print(f"Max retries exceeded for {xxx}, skipping...")
                continue
                
        except RequestException as e:
            print(f"Request failed with error: {e}")
            retry_count += 1
            if retry_count < max_retries:
                time.sleep(30)
    
    if r is None or r.status_code != 200:
        print(f"Failed to download {xxx} after {max_retries} attempts, skipping...")
        continue
    
    print(r)
    
    foo = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

    foo.columns = r.json()[0]

    pq.write_table(pa.Table.from_pandas(foo), out_file)


<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
Connection timeout on attempt 1/5: ConnectTimeout
Waiting 30 seconds before retry...
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>


In [9]:
# Append dated files to current files and replace them
import glob

data_dir = ".\\data\\imports-hs10"

# Find all current files
current_files = glob.glob(os.path.join(data_dir, "*data-current.parquet"))

for current_file in current_files:
    # Extract the country code from the filename
    filename = os.path.basename(current_file)
    country_code = filename.replace("data-current.parquet", "")
    
    # Find the corresponding dated file
    dated_file = os.path.join(data_dir, f"{country_code}data-{date}.parquet")
    
    if not os.path.exists(dated_file):
        print(f"Dated file not found for {country_code}, skipping...")
        continue
    
    print(f"Processing {country_code}...")
    
    # Read both files
    current_df = pq.read_table(current_file).to_pandas()
    dated_df = pq.read_table(dated_file).to_pandas()
    
    # Append dated data to current
    combined_df = pd.concat([current_df, dated_df], ignore_index=True)
    
    # Write back to current file
    pq.write_table(pa.Table.from_pandas(combined_df), current_file)
    
    print(f"Updated {country_code}: appended {len(dated_df)} rows, total now {len(combined_df)}")

print("Done!")


Processing 0003...
Updated 0003: appended 16020 rows, total now 2224662
Processing 0020...
Updated 0020: appended 14012 rows, total now 1863998
Processing 1220...
Updated 1220: appended 12274 rows, total now 1580652
Processing 2010...
Updated 2010: appended 10151 rows, total now 1300305
Processing 3010...
Updated 3010: appended 4067 rows, total now 392228
Processing 3370...
Updated 3370: appended 1823 rows, total now 189949
Processing 3510...
Updated 3510: appended 6066 rows, total now 688883
Processing 4010...
Updated 4010: appended 5011 rows, total now 589176
Processing 4120...
Updated 4120: appended 10555 rows, total now 1325421
Processing 4190...
Updated 4190: appended 3098 rows, total now 330927
Processing 4210...
Updated 4210: appended 7012 rows, total now 818632
Processing 4231...
Updated 4231: appended 5646 rows, total now 669100
Processing 4279...
Updated 4279: appended 9877 rows, total now 1239554
Processing 4280...
Updated 4280: appended 11009 rows, total now 1463570
Process

In [13]:
combined_df.CTY_NAME.unique()

array(['TOTAL FOR ALL COUNTRIES'], dtype=object)

In [14]:
# Combine all current data files into one big dataset
all_data = []

for current_file in current_files:
    # Extract the country code from the filename
    filename = os.path.basename(current_file)
    country_code = filename.replace("data-current.parquet", "")
    
    print(f"Reading {country_code}...")
    
    # Read the file
    df_country = pq.read_table(current_file).to_pandas()
    
    # Add country code column if not already present
    if 'CTY_CODE' not in df_country.columns:
        df_country['CTY_CODE'] = country_code
    
    all_data.append(df_country)

# Combine all countries into one dataframe
bigdf = pd.concat(all_data, ignore_index=True)

print(f"\nCombined dataset: {len(bigdf):,} rows, {len(all_data)} countries")

# Save to parquet
output_file = ".\\data\\imports-hs10\\ALL-data-current.parquet"
pq.write_table(pa.Table.from_pandas(bigdf), output_file)
print(f"Saved to: {output_file}")

Reading 0003...
Reading 0020...
Reading 1220...
Reading 2010...
Reading 3010...
Reading 3370...
Reading 3510...
Reading 4010...
Reading 4120...
Reading 4190...
Reading 4210...
Reading 4231...
Reading 4279...
Reading 4280...
Reading 4330...
Reading 4419...
Reading 4621...
Reading 4700...
Reading 4759...
Reading 5081...
Reading 5170...
Reading 5330...
Reading 5490...
Reading 5520...
Reading 5570...
Reading 5590...
Reading 5600...
Reading 5700...
Reading 5800...
Reading 5830...
Reading 5880...
Reading 6021...
Reading TOTAL...

Combined dataset: 32,386,738 rows, 33 countries
Saved to: .\data\imports-hs10\ALL-data-current.parquet


In [16]:
bigdf.CTY_NAME.unique()

array(['EUROPEAN UNION', 'USMCA (NAFTA)', 'CANADA', 'MEXICO', 'COLOMBIA',
       'CHILE', 'BRAZIL', 'SWEDEN', 'UNITED KINGDOM', 'IRELAND',
       'NETHERLANDS', 'BELGIUM', 'FRANCE', 'GERMANY', 'AUSTRIA',
       'SWITZERLAND', 'RUSSIA', 'SPAIN', 'ITALY', 'ISRAEL',
       'SAUDI ARABIA', 'INDIA', 'THAILAND', 'VIETNAM', 'MALAYSIA',
       'SINGAPORE', 'INDONESIA', 'CHINA', 'KOREA, SOUTH', 'TAIWAN',
       'JAPAN', 'AUSTRALIA', 'TOTAL FOR ALL COUNTRIES'], dtype=object)